<a href="https://colab.research.google.com/github/srivatsan88/YouTubeLI/blob/master/TPOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Link to supporting video for below notebook walkhthrough - https://www.youtube.com/watch?v=oIFSpnFGX8s





In [ ]:
!pip install tpot

![alt text](https://raw.githubusercontent.com/EpistasisLab/tpot/master/images/tpot-ml-pipeline.png)

**Import TPOT packages and load Telecom Churn dataset**

In [ ]:
from tpot import  TPOTClassifier
from sklearn.model_selection import train_test_split
from __future__ import print_function
import sys,tempfile, urllib, os
import pandas as pd
import numpy as np

In [ ]:
BASE_DIR = '/tmp'
OUTPUT_FILE = os.path.join(BASE_DIR, 'churn_data.csv')

churn_data=urllib.request.urlretrieve('https://raw.githubusercontent.com/srivatsan88/YouTubeLI/master/dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv', OUTPUT_FILE)

In [ ]:
churn_df = pd.read_csv(OUTPUT_FILE)

In [ ]:
churn_df.head()

In [ ]:
print ("Rows     : " ,churn_df.shape[0])
print ("Columns  : " ,churn_df.shape[1])

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder

categorical_columns = ['gender', 'Partner', 'Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Churn']
column_trans = make_column_transformer((OrdinalEncoder(), categorical_columns))

churn_transformed=column_trans.fit_transform(churn_df)

In [ ]:
churn_df_trans = churn_df.copy()
churn_df_trans = pd.DataFrame(churn_transformed, columns=categorical_columns)

In [ ]:
churn_df.update(churn_df_trans)

In [ ]:
churn_df

In [ ]:
churn_df.isna().sum()

In [ ]:
churn_df.replace(r'^\s*$', np.nan, regex=True).isna().sum()

In [ ]:
churn_df = churn_df.replace(r'^\s*$', np.nan, regex=True)

In [ ]:
churn_df.isna().sum()

In [ ]:
churn_df.dtypes

In [ ]:
churn_df.iloc[:, 19] = pd.to_numeric(churn_df.iloc[:, 19], errors='coerce') 

In [ ]:
from sklearn.impute import SimpleImputer
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')

In [ ]:
churn_df.iloc[:, 19] = imp_median.fit_transform(churn_df.iloc[:, 19].values.reshape(-1, 1) )

In [ ]:
churn_df.isna().sum()

In [ ]:
churn_df_X = churn_df.drop("Churn", axis=1)
churn_df_X = churn_df_X.drop("customerID", axis=1)
churn_df_y = churn_df['Churn']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(churn_df_X, churn_df_y, train_size=0.75, test_size=0.25)

In [ ]:
X_train

**Use TPOT classifier to find the best model algorithm that fits the data. TPOT uses genetic algorithm based search internally**


In [ ]:
tpot =  TPOTClassifier(generations=4, population_size=10, verbosity=3)
tpot.fit(X_train, y_train)
print(tpot.score(X_test, y_test))

In [ ]:
import multiprocessing

if __name__ == '__main__':
    multiprocessing.set_start_method('forkserver', force=True)
    tpot =  TPOTClassifier(generations=2, population_size=20, verbosity=2,n_jobs = 20, random_state=50)
    tpot.fit(X_train, y_train)


In [ ]:
print(tpot.score(X_test, y_test))

**Export the scikit learn code for best classifier and dispay the code**

In [ ]:
tpot.export('tpot_churn_pipeline.py')

In [ ]:
!cat tpot_churn_pipeline.py

**Get details on algorithm TPOT searched for and how the performed on the dataset**



In [ ]:
tpot.evaluated_individuals_

In [ ]:
tpot.fitted_pipeline_